# Z event generation

Sample generation for fast detector simulation input

### $e^+e^- \to Z^*/\gamma \to \mu^+\mu^-$


## Setup the environment

In [1]:
#General utility libraries
import os, sys, subprocess, time

#General data operations library
import math
import numpy as np

#HEP specific tools
import scipy.constants as scipy_constants
from particle import Particle
import ROOT

#Plotting libraries
import matplotlib.pyplot as plt

#Increase plots font size
params = {'legend.fontsize': 'xx-large',
          'figure.figsize': (10, 7),
         'axes.labelsize': 'xx-large',
         'axes.titlesize':'xx-large',
         'xtick.labelsize':'xx-large',
         'ytick.labelsize':'xx-large'}
plt.rcParams.update(params)

#Helper function to supress Pythia messages
def makePythiaSilent(pythia):
    pythia.ReadString("Init:showMultipartonInteractions = off")
    pythia.ReadString("Init:showChangedParticleData = off")
    pythia.ReadString("Init:showProcesses = off")
    pythia.ReadString("Init:showChangedSettings = off")
    pythia.ReadString("Next:numberShowInfo = 0")
    pythia.ReadString("Next:numberShowProcess = 0")
    pythia.ReadString("Next:numberShowEvent = 0")
    pythia.ReadString("Next:numberCount = 0")

/scratch1/gjedrzej/.local/lib/python3.9/site-packages/ROOT/__init__.py:5: UserWarning: 
This distribution of ROOT is in alpha stage. Feedback is welcome and appreciated. Feel free to reach out to the user forum for questions and general feedback at https://root-forum.cern.ch or to submit an issue at https://github.com/root-project/root/issues. Do not rely on this distribution for production purposes.

  warnings.warn(
/scratch1/gjedrzej/.local/lib/python3.9/site-packages/cppyy/__init__.py:374: UserWarning: CPyCppyy API not found (tried: /usr/include/site/python3.9); set CPPYY_API_PATH envar to the 'CPyCppyy' API directory to fix
  warnings.warn("CPyCppyy API not found (tried: %s); "


# Run Pythia8 from ROOT interface

Prepare and initialize pythia configuration for generation of $e^+e^- \to Z^*/\gamma \to \mu^+\mu^-$ events at 240 GeV

 * create `TPythia8` object managing the Pythia
 * enable corresponding production process
 * initialize pythia to simulate $e^+e^-$ collisions at 240 GeV
 * fix random number generator seed to a fixed value to get the same results every time

In [2]:
#Create Pythia control object
pythia = ROOT.TPythia8(False)

#Disable messages
makePythiaSilent(pythia)

#Set configurations parameters
pythia.ReadString("Random:setSeed = on")
pythia.ReadString("Random:seed = 99")

#Set the initial particles PDG id and energy in center of mass system in GeV

electronPDGId = int(Particle.from_evtgen_name("e-").pdgid)
positonPDGId  = int(Particle.from_evtgen_name("e+").pdgid)

pdgId_beam1 = electronPDGId
pdgId_beam2 = positonPDGId

sqrtS = 240 #GeV

AttributeError: Failed to get attribute TPythia8 from ROOT

In [3]:
# Our proces label: to be used for output file name

proc = "Zmumu"

# Number of events to generate

Nevt = 10000

# Select PYTHIA process of interest (default all processes are switched off).

pythia.ReadString("WeakSingleBoson:ffbar2gmZ = on") 

# Select decay channel(s) for Z - start with only muon decays

Z0PDGId       = int(Particle.from_evtgen_name("Z0").pdgid)

muonPDGId  = int(Particle.from_evtgen_name("mu-").pdgid)

pythia.ReadString(str(Z0PDGId)+":onMode = off")                 # Switch of all decays
pythia.ReadString(str(Z0PDGId)+":onIfAny = "+str(muonPDGId))    # Switch on only the selected ones


In [4]:
# Initialize Pythia

pythia.Initialize(pdgId_beam1 , pdgId_beam2, sqrtS)

True

 PYTHIA Warning in PhaseSpace::trialKin123: maximum for cross section violated


In [5]:
#Prepare space for list of final and intermediate state particles
nParticles = 1000
particles = ROOT.TClonesArray("TParticle", nParticles)

In [6]:
%%time
# Generate single event and count final state particles
pythia.GenerateEvent()
pythia.EventListing()

pythia.ImportParticles(particles,"All")
print("There are {} particles in the first event history".format(particles.GetEntries()))

There are 13 particles in the first event history
CPU times: user 82.2 ms, sys: 4.92 ms, total: 87.1 ms
Wall time: 87.9 ms

 --------  PYTHIA Event Listing  (complete event)  ---------------------------------------------------------------------------------
 
    no         id  name            status     mothers   daughters     colours      p_x        p_y        p_z         e          m 
     0         90  (system)           -11     0     0     0     0     0     0      0.000      0.000      0.000    240.000    240.000
     1         11  (e-)               -12     0     0     6     0     0     0      0.000      0.000    120.000    120.000      0.001
     2        -11  (e+)               -12     0     0     7     0     0     0      0.000      0.000   -120.000    120.000      0.001
     3         11  (e-)               -21     6     0     5     0     0     0      0.000      0.000     16.723     16.723      0.000
     4        -11  (e+)               -21     7     7     5     0     0     0 

In [7]:
# Event consistency check

tot4mom = ROOT.TLorentzVector(0.,0.,0.,0.)    

for part in particles:
    if part.GetStatusCode()==1 :  
        tot4mom += ROOT.TLorentzVector(part.Px(), part.Py(), part.Pz(), part.Energy())    

print("Invariant mass of the final state particles: {:.5f} GeV".format(tot4mom.M()))

Invariant mass of the final state particles: 240.00000 GeV


In [8]:
# Load delphes libraries 

ROOT.gInterpreter.AddIncludePath('/opt/Delphes-3.5.0/')
ROOT.gInterpreter.AddIncludePath('/opt/Delphes-3.5.0/external')
ROOT.gInterpreter.Declare('#include "classes/DelphesClasses.h"')
ROOT.gInterpreter.Declare('#include "ExRootAnalysis/ExRootTreeBranch.h"')
ROOT.gInterpreter.Declare('#include "ExRootAnalysis/ExRootTreeReader.h"')
ROOT.gInterpreter.Declare('#include "ExRootAnalysis/ExRootTreeWriter.h"')

ROOT.gSystem.Load("/opt/Delphes-3.5.0/libDelphes")

0

In [9]:
# Open output file and create the required tree structure 

fname = proc + "_pythia.root"
evtFile = ROOT.TFile(fname,"recreate")
treeWriter = ROOT.ExRootTreeWriter(evtFile, "Delphes");

branchEvent = treeWriter.NewBranch("Event", ROOT.HepMCEvent.Class());
branchParticle = treeWriter.NewBranch("Particle", ROOT.GenParticle.Class());

In [10]:
%%time

pythia8 = pythia.Pythia8()   # Actual pointer to Pythia8 (internal in TPythia8)

# Event generation and storing

for ievt in range(Nevt):

    pythia.GenerateEvent()
    particles.Clear()
    pythia.ImportParticles(particles,"All")

    # Event header
    
    Event = branchEvent.NewEntry()
    
    Event.Number = ievt+1

    # Copy pythia event information
    
    Event.ProcessID = pythia8.info.code()
    Event.Weight = pythia8.info.weight()

    # Copy particles
    
    for part in particles:
        
        pid = part.GetPdgCode()

        # Skip pythia documentation lines
        
        if pythia8.particleData.isParticle(pid) :

            Particle = branchParticle.NewEntry()
    
            Particle.PID = pid
            Particle.Status = part.GetStatusCode() 
            Particle.IsPU = 0
            
            Particle.E  = part.Energy()
            Particle.Px = part.Px()
            Particle.Py = part.Py()
            Particle.Pz = part.Pz()
    
            parP4 = ROOT.TLorentzVector(part.Px(),part.Py(),part.Pz(),part.Energy())
    
            Particle.P = parP4.E()
            Particle.PT = parP4.Pt()
            Particle.Eta = parP4.Eta()
            Particle.Phi = parP4.Phi()
            Particle.Rapidity = parP4.Rapidity()
            
            Particle.X = part.Vx()
            Particle.Y = part.Vy()
            Particle.Z = part.Vz()
            Particle.T = part.T()
    
            Particle.M1 = part.GetFirstMother()
            Particle.M2 = part.GetSecondMother()
    
            Particle.D1 = part.GetFirstDaughter()
            Particle.D2 = part.GetLastDaughter()
    
            Particle.Charge = int(pythia8.particleData.charge(pid))
            Particle.Mass = pythia8.particleData.m0(pid)
        
    treeWriter.Fill()
    treeWriter.Clear()

CPU times: user 6.26 s, sys: 917 ms, total: 7.18 s
Wall time: 7.2 s


In [11]:
treeWriter.Write()
evtFile.Close()

## Running Delphes

To process Pythia events generated with the fast simulation framwork you need to open terminal window (can be done with JupyterLab launcher) and execute: <br><br>

<tt>/opt/Delphes-3.5.0/DelphesROOT /opt/Delphes-3.5.0/cards/delphes_card_ILCgen.tcl Zmumu_delphes.root Zmumu_pythia.root </tt><br>

Where the Delphes programm (<tt>DelphesROOT</tt>) take the following arguments:
  * detector model cards file name
  * fast simulation output file name
  * name(s) of the input (Pythia) file(s)

For this workshop we will use ILC generic detector model: 
https://github.com/iLCSoft/ILCDelphes/

